# Quickstart

Every recorder stream is a DuckDB view (see `docs/RECORDER.md` for columns and `analysis/README.md` for the
derived views). Run `uv run polydata update` first to sync and build the data.

In [ ]:
import polydata
import matplotlib.pyplot as plt

con = polydata.connect()
con.sql("SELECT table_name FROM information_schema.tables ORDER BY 1").df().T

## Markets
One row per Up/Down market: window, token ids, fees, `price_to_beat`, `outcome`.

In [ ]:
con.sql("""
    SELECT timeframe, count(*) AS markets, count(outcome) AS resolved,
           round(avg((outcome = 'Up')::INT), 3) AS up_rate
    FROM markets GROUP BY 1 ORDER BY min(window_end_ms - window_start_ms)
""").df()

## One 5-minute market: Polymarket vs Chainlink

Pick the most traded resolved BTC 5m market and plot the Up token's mid price next to the Chainlink price
(the resolution source) and the price to beat.

In [ ]:
m = con.sql("""
    SELECT m.slug, m.window_start_ms, m.window_end_ms, m.price_to_beat, m.outcome
    FROM markets m JOIN pm_trade t USING (slug)
    WHERE m.coin = 'btc' AND m.timeframe = '5m' AND m.outcome IS NOT NULL
    GROUP BY ALL ORDER BY count(*) DESC LIMIT 1
""").fetchall()[0]
slug, start_ms, end_ms, ptb, outcome = m
print(slug, outcome, ptb)

up = con.sql(f"""
    SELECT ms(ts_ms) AS t, (best_bid + best_ask) / 2 AS mid FROM pm_best_bid_ask
    WHERE slug = '{slug}' AND outcome = 'Up' AND ts_ms BETWEEN {start_ms} - 60000 AND {end_ms}
    ORDER BY ts_ms
""").df()
cl = con.sql(f"""
    SELECT ms(ts_ms) AS t, price FROM chainlink
    WHERE coin = 'btc' AND ts_ms BETWEEN {start_ms} - 60000 AND {end_ms} ORDER BY ts_ms
""").df()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(up.t, up.mid, drawstyle='steps-post', label='Up mid (Polymarket)')
ax.set_ylabel('Up probability'); ax.set_ylim(0, 1)
ax2 = ax.twinx()
ax2.plot(cl.t, cl.price, color='tab:orange', label='Chainlink BTC/USD')
ax2.axhline(ptb, color='tab:orange', ls='--', lw=1, label='price to beat')
fig.legend(loc='upper left'); ax.set_title(f'{slug} -> {outcome}')
plt.show()

## As-of join: Binance price at the moment of every Polymarket trade

`ASOF JOIN` matches each row with the latest row of the other table at or before its timestamp, which is
the building block for backtests (what did we know when?). Join on `recv_ms` (when *we* saw it) to avoid
lookahead.

In [ ]:
con.sql(f"""
    SELECT t.recv_ms, t.outcome, t.side, t.price, t.size,
           (b.bid + b.ask) / 2 AS binance_mid, t.recv_ms - b.recv_ms AS binance_age_ms
    FROM (SELECT * FROM pm_trade WHERE slug = '{slug}') t
    ASOF JOIN (SELECT * FROM bn_fut_bbo WHERE sym = 'BTCUSDT'
               AND recv_ms BETWEEN {start_ms} - 120000 AND {end_ms}) b
      ON t.recv_ms >= b.recv_ms
    ORDER BY t.recv_ms LIMIT 10
""").df()

## Rebuild `price_to_beat` from Chainlink

`priceToBeat` = mean Chainlink price over the 60 s before the window opens. Comparing our reconstruction
with Polymarket's value checks both the theory and our Chainlink coverage.

In [ ]:
con.sql("""
    WITH r AS (
        SELECT m.slug, m.coin, m.price_to_beat, avg(c.price) AS ours, count(*) AS ticks
        FROM markets m JOIN chainlink c
          ON c.coin = m.coin AND c.ts_ms >= m.window_start_ms - 60000 AND c.ts_ms < m.window_start_ms
        WHERE m.price_to_beat IS NOT NULL
        GROUP BY ALL
    )
    SELECT coin, count(*) AS markets,
           round(median(abs(ours / price_to_beat - 1)) * 1e4, 3) AS median_err_bp,
           round(quantile_cont(abs(ours / price_to_beat - 1), 0.99) * 1e4, 3) AS p99_err_bp,
           min(ticks) AS min_ticks
    FROM r GROUP BY 1 ORDER BY 1
""").df()

## Who trades? Data API trades include the wallet

In [ ]:
con.sql("""
    SELECT proxy_wallet, any_value(pseudonym) AS pseudonym, count(*) AS trades,
           count(DISTINCT slug) AS markets, round(sum(price * size)) AS usdc
    FROM pm_api_trades GROUP BY 1 ORDER BY usdc DESC LIMIT 10
""").df()